<!-- nav -->
[← 6 · Analyzing and rewriting constraints](06_Analyzing_And_Rewriting_Constraints.ipynb) · [Home](../../README.md) · [8 · Constraints for firmware and hardware →](08_Constraints_For_Firmware_And_Hardware.ipynb)

# 7 · One constraint, many languages

**The problem.** Back to the four copies of case study 1. The QA team checks shipments in Excel; the data pipeline is
Python; the analysts use MATLAB; the specification is a LaTeX document. Each wants the constraint in its own language,
and each copy, written by hand, drifts.

mbse-expressions models each of those languages as a **dialect**: its expressions are data too, it has its own
evaluator, and it writes its own source text. **Translators** between dialects turn the neutral constraint (in the Basic
dialect, which the earlier case studies used) into each, and back.

In [1]:
import math

from mbse.Expressions import Evaluators, Expressions as E, Translators
from mbse.Expressions.Dialects import Excel, Latex, Matlab, Python
from mbse.Expressions.Framework import Terms
from toolkit import cold_chain, show

store = cold_chain()  # with each dialect's kinds installed, so that it holds their expressions too
Expression, Term = store.Expressions.Framework.Expression, store.Expressions.Framework.Term
shipment, c = E.variable("shipment"), E.variable("c")
with store.transaction():
    in_range = Expression(lambda e: e.root(E.literal(2.0).le(shipment.celsius).and_(shipment.celsius.le(8.0))))
    allowed = Expression(lambda e: e.root(shipment.readings.count_where("c", c.gt(8.0)).le(2).and_(shipment.readings.max().le(12.0))))
languages = [(Excel.Expressions.DIALECT, Excel.Text), (Python.Expressions.DIALECT, Python.Text),
             (Matlab.Expressions.DIALECT, Matlab.Text), (Latex.Expressions.DIALECT, Latex.Text)]


def translate(constraint, source, target):
    """`constraint`, of the dialect `source`, translated to `target`: the translation is a writer, written as a term."""
    with store.transaction():
        return Term(Translators.between(source, target).forward(constraint))

## Step 1: translating

`Translators.between(source, target)` finds the translator, and `forward` translates, giving a writer of the target
dialect's term, which `translate` (above) writes in a transaction, as any term is written. Each dialect's `Text.ToText`
writes its source:

In [2]:
for dialect, text in languages:
    print(f"{dialect.name():7}", text.ToText(translate(in_range, E.DIALECT, dialect)))

Excel   =AND(2.0 <= shipment.celsius, shipment.celsius <= 8.0)
Python  2.0 <= shipment.celsius and shipment.celsius <= 8.0
Matlab  2.0 <= shipment.celsius && shipment.celsius <= 8.0
Latex   2.0 \leq \mathit{shipment}.\mathit{celsius} \land \mathit{shipment}.\mathit{celsius} \leq 8.0


Collections translate to each language's idiom: `MAP` and `LAMBDA` in Excel, generator expressions in Python,
`arrayfun` in MATLAB. LaTeX has no counterpart for a quantifier, and says so:

In [3]:
for dialect, text in languages:
    try:
        print(f"{dialect.name():7}", text.ToText(translate(allowed, E.DIALECT, dialect)))
    except ValueError as error:
        print(f"{dialect.name():7}", error)

Excel   =AND(SUM(MAP(shipment.readings, LAMBDA(c, IF(c > 8.0, 1, 0)))) <= 2, MAX(shipment.readings) <= 12.0)
Python  sum(1 for c in shipment.readings if c > 8.0) <= 2 and max(shipment.readings) <= 12.0
Matlab  nnz(arrayfun(@(c) c > 8.0, shipment.readings)) <= 2 && max(shipment.readings) <= 12.0
Latex   Basic quantifier 'count' has no Latex counterpart


## Step 2: each language evaluates by its own rules

A translation keeps the constraint's **form**. Its value is the target language's business, because that's where it will
run. Evaluate the translations on a shipment whose logger never reported:

In [4]:
with store.transaction():
    silent = store.Shipment(lambda s: s.id("S3"))
excel = translate(in_range, E.DIALECT, Excel.Expressions.DIALECT)
python = translate(in_range, E.DIALECT, Python.Expressions.DIALECT)
print("Basic  ", Evaluators.OfAny(in_range, {"shipment": silent}))
print("Excel  ", Excel.Evaluators.OfAny(excel, {"shipment": silent}))
try:
    Python.Evaluators.OfAny(python, {"shipment": silent})
except AttributeError as error:
    print("Python ", type(error).__name__, error)

Basic   None
Excel   #FIELD!
Python  AttributeError 'Shipment' object has no attribute 'celsius'


Basic says unknown; Excel gives the error value `#FIELD!`, which propagates through a worksheet the way errors do
there; Python raises, as `shipment.celsius` would. Coercion differs too. Excel compares an integer limit with a float
reading happily, where Basic says unknown (case study 2):

In [5]:
with store.transaction():
    sloppy = Expression(lambda e: e.root(shipment.celsius.ge(2)))
    cold = store.Shipment(lambda s: s.id("S1").celsius(5.5))
print("Basic  ", Evaluators.OfAny(sloppy, {"shipment": cold}))
print("Excel  ", Excel.Evaluators.OfAny(translate(sloppy, E.DIALECT, Excel.Expressions.DIALECT), {"shipment": cold}))

Basic   None
Excel   True


So keep the constraint in Basic, where its meaning is fixed, translate it to where it is checked, and test it there with
that language's evaluator, which models its rules exactly.

## Step 3: reading constraints back

A constraint written in another language can be read into Basic, to analyze it (case study 6) or store it. In Python,
`Text.FromText` parses source into a writer of the Python dialect's term, and the translator goes the other way:

In [6]:
source = "2.0 <= shipment.celsius <= 8.0"
with store.transaction():
    parsed = Term(Python.Text.FromText(source))
back = translate(parsed, Python.Expressions.DIALECT, E.DIALECT)
print(show(back))
Terms.same(back, in_range)

and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0))


True

Round trips keep what both languages can say. What one of them can't, such as a let in MATLAB, which has none, is
written another way, and comes back in that other form.

## Step 4: what a constraint may reach

A Python or MATLAB expression can call functions, and an expression loaded from a file mustn't be able to call
anything it likes. Each dialect's scope reaches only what the caller gives it: here, Python's `math`, and nothing
else:

In [7]:
with store.transaction():
    rounded = Expression(lambda e: e.root(Python.Text.FromText("import math\nmath.floor(shipment.celsius) <= 8")))
print(Python.Text.ToText(rounded))
try:
    Python.Evaluators.OfAny(rounded, {"shipment": cold})
except ImportError as error:
    print(type(error).__name__, error)
Python.Evaluators.OfAny(rounded, Python.Evaluators.Scope({"shipment": cold}, modules={"math": math}))

import math
math.floor(shipment.celsius) <= 8
ImportError import of 'math' is not allowed by the scope


True

## Why it works this way

- **One translator per pair, by rules.** Each pair of dialects has its own translator, declared as rules pairing
  patterns, as the rewriting of case study 6 was. Translating directly or through Basic gives the same result.
- **Form, not value.** A translator can't make Excel treat a missing value as unknown, and shouldn't try: the
  translated constraint must read the way an engineer in that language would write it.
- **Nothing is reachable by default.** Constraints travel as data, sometimes from people you don't fully trust. A scope
  is an allowlist.

<!-- nav -->
[← 6 · Analyzing and rewriting constraints](06_Analyzing_And_Rewriting_Constraints.ipynb) · [Home](../../README.md) · [8 · Constraints for firmware and hardware →](08_Constraints_For_Firmware_And_Hardware.ipynb)